In [106]:
import requests_async
import os
import math
import time
import pandas as pd

In [107]:
repositories = set()

In [112]:
search_terms = [
    # Maven
    'language:xml "<artifactId>jooq</artifactId>"',
    'language:xml "<artifactId>jooq-"',
    'language:xml "<artifactId>jooq-codegen-maven</artifactId>"',
    'language:xml "<artifactId>jooq-codegen</artifactId>"',
    'language:xml "<artifactId>jooq-meta/artifactId>"',
    'language:xml "<artifactId>jooq-meta-extensions</artifactId>"',
    'language:xml "<artifactId>jooq-meta-extensions-liquibase</artifactId>"',
    'language:xml "<artifactId>jooq-meta-extensions-hibernate</artifactId>"',
    'language:xml "<artifactId>jooq-postgres-extensions</artifactId>"',
    'language:xml "<artifactId>jooq-jackson-extensions</artifactId>"',
    'language:xml "<artifactId>jooq-jpa-extensions</artifactId>"',
    'language:xml "<artifactId>jooq-reactor-extensions</artifactId>"',
    'language:xml "<artifactId>jooq-beans-extensions</artifactId>"',
    'language:xml "<artifactId>jooq-checker</artifactId>"',

    'language:"Maven POM" "<artifactId>jooq</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-"',
    'language:"Maven POM" "<artifactId>jooq-codegen-maven</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-codegen</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-meta</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-meta-extensions</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-meta-extensions-liquibase</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-meta-extensions-hibernate</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-postgres-extensions</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-jackson-extensions</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-jpa-extensions</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-reactor-extensions</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-beans-extensions</artifactId>"',
    'language:"Maven POM" "<artifactId>jooq-checker</artifactId>"',
    
    # Gradle short syntax
    'language:gradle "\\"org.jooq:jooq:"',
    'language:gradle "\'org.jooq:jooq:"',
    'language:gradle "\\"org.jooq:jooq-"',
    'language:gradle "\'org.jooq:jooq-"',
    'language:gradle "\\"org.jooq:jooq-meta"',
    'language:gradle "\'org.jooq:jooq-meta"',
    'language:gradle "\\"org.jooq:jooq"',
    'language:gradle "\'org.jooq:jooq"',

    # Gradle long syntax
    'language:gradle "group: \\"org.jooq\\""',
    'language:gradle "group:\\"org.jooq\\""',
    'language:gradle "group: \'org.jooq\'"',
    'language:gradle "group:\'org.jooq\'"',

    # Gradle plugins
    'language:gradle "jooq-codegen-gradle"',
    'language:gradle "\\"nu.studer.jooq\\""',
    'language:gradle "\'nu.studer.jooq\'"',

    # Gradle Kotlin syntax
    'extension:kts "org.jooq:jooq:"',
    'extension:kts "org.jooq:jooq-"',
    'extension:kts "org.jooq:jooq"',
    'extension:kts "jooq-codegen-gradle"',
    'extension:kts "nu.studer.jooq"',
]

In [113]:
async def request(search_term, page):
    return await requests_async.get(
        'https://api.github.com/search/code', 
        params = {'q': search_term, 'per_page': 100, 'page': page}, 
        headers = {'Authorization': 'Bearer ' + os.getenv('GITHUB_TOKEN'), 'X-GitHub-Api-Version': '2022-11-28'}
    )

for search_term in search_terms:
    page = 1
    while True:
        time.sleep(6.5)
        response = await request(search_term, page)
        total_count = response.json()['total_count']

        for file in response.json()['items']:
            repositories.add(file['repository']['full_name'])

        print(f"After search term {search_term}, page {page}: {len(repositories)}")

        page += 1
        if page > int(math.ceil(min(total_count, 1000) / 100.0)):
            break

After search term language:gradle "\"nu.studer.jooq\"", page 1: 3827
After search term language:gradle "'nu.studer.jooq'", page 1: 3829
After search term language:gradle "'nu.studer.jooq'", page 2: 3829
After search term language:gradle "'nu.studer.jooq'", page 3: 3829
After search term language:gradle "'nu.studer.jooq'", page 4: 3829
After search term language:gradle "'nu.studer.jooq'", page 5: 3829
After search term language:gradle "'nu.studer.jooq'", page 6: 3829
After search term language:gradle "'nu.studer.jooq'", page 7: 3829
After search term language:gradle "'nu.studer.jooq'", page 8: 3829
After search term language:gradle "'nu.studer.jooq'", page 9: 3829
After search term language:gradle "'nu.studer.jooq'", page 10: 3829


In [114]:
len(repositories)

3829

In [116]:
repositories_df = pd.DataFrame(list(repositories), columns=['name'])
repositories_df = repositories_df.sort_values(by=['name'])
repositories_df.to_csv('../datasets/all-repositories.csv', index=False)